# CS383: Data Science and Machine Learning
## Week 7 Sneak Peek — The Whole ML Workflow, on Training Wheels

*Dr. Thitima Srivatanakul*

Every other lecture this semester uses live, real NYC data on purpose — it's messier, it's more honest, and
it's what the job actually looks like. Today we're breaking that rule, once, deliberately.

### Why break the rule, just this once?

We're about to spend the next several weeks building real machine learning skills: training a model,
evaluating it honestly, comparing model types, tuning it. That's a lot of new ideas at once. If we tried to
learn all of them for the first time *and* fight messy real-world data at the same time, it would be hard to
tell which part was confusing — the concept, or the data.

So today we use the **Iris flower dataset** — 150 flowers, 3 species, 4 measurements each, no missing
values, perfectly balanced, collected in 1936. It is the single most famous toy dataset in all of machine
learning, for exactly the reasons your course landing page warns you about: every hard decision has already
been made for you. That's a bad way to learn what real data science is like — but a genuinely good way to
see the *shape* of the ML workflow for the first time, uncluttered.

**This is the only lecture all semester that uses Iris.** Treat today as a movie trailer for Weeks 7–11:
train/test splits, evaluation metrics, cross-validation, and comparing model types. Everything after today
goes right back to real, messy NYC data — and you'll notice it's harder. That's the point.

---

## Part 1 — Meet the (Toy) Dataset

Each row is one iris flower. Four measurements (in centimeters), and a species label: *setosa*,
*versicolor*, or *virginica*.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

iris = load_iris()
X = iris.data
y = iris.target

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = [iris.target_names[label] for label in iris.target]

df.sample(10)

In [ ]:
print(df.shape)
print(df["species"].value_counts())
df.describe()

Notice `value_counts()` — exactly 50 flowers of each species. Real data is never this balanced by
accident. Keep that in the back of your mind; it'll matter later this semester when accuracy numbers on
real, imbalanced data turn out to be misleading (Week 9).

### A quick look before modeling anything — same EDA habits from Week 5

In [ ]:
sns.histplot(data=df, x="sepal length (cm)", hue="species", kde=True)
plt.title("Distribution of Sepal Length")
plt.show()

In [ ]:
sns.scatterplot(data=df, x="petal length (cm)", y="petal width (cm)", hue="species", s=80)
plt.title("Petal Length vs. Petal Width")
plt.show()

Look how cleanly the three species separate just on petal measurements. That's the toy-dataset effect in
action — real classification problems are rarely this visually obvious.

In [ ]:
sns.pairplot(df, hue="species", corner=True)
plt.show()

In [ ]:
sns.boxplot(data=df, x="species", y="petal length (cm)")
plt.title("Petal Length by Species")
plt.show()

---

## Part 2 — Framing This as a Classification Problem

We want to predict **species** — a category, not a number. That makes this a **classification** task. The
target (`y`) is categorical; the features (`X`) are the four measurements.

Just like every regression/classification example this semester, step one is splitting the data into a
training set (the model learns from this) and a test set (we evaluate on this — data the model has never
seen).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,  # keep the same species proportions in both train and test
)

print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

Now we build and train (`.fit()`) a logistic regression model — despite the name, logistic regression is
a **classification** algorithm, not a regression one. (We'll untangle that naming confusion in Part 5.)

In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:     ", y_test)

### Evaluating the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=iris.target_names, cmap="Blues"
)
plt.show()

The confusion matrix is a preview of Week 9. For now, just read it as: rows are what the flower actually
was, columns are what the model guessed, and the diagonal is where it got it right. Everything off the
diagonal is a specific kind of mistake worth naming, not just a single "wrong" count.

---

## Part 3 — Does It Matter Which Features We Use?

So far we used all four measurements. What if we only gave the model petal measurements? Or only sepal
measurements?

In [ ]:
X_petal = df[["petal length (cm)", "petal width (cm)"]]

X_train_petal, X_test_petal, y_train_petal, y_test_petal = train_test_split(
    X_petal, y, test_size=0.20, random_state=42, stratify=y
)

model_petal = LogisticRegression(max_iter=200)
model_petal.fit(X_train_petal, y_train_petal)
y_pred_petal = model_petal.predict(X_test_petal)

print("Petal-only accuracy:", accuracy_score(y_test_petal, y_pred_petal))
print("Petal-only confusion matrix:")
print(confusion_matrix(y_test_petal, y_pred_petal))

In [ ]:
X_sepal = df[["sepal length (cm)", "sepal width (cm)"]]

X_train_sepal, X_test_sepal, y_train_sepal, y_test_sepal = train_test_split(
    X_sepal, y, test_size=0.20, random_state=42, stratify=y
)

model_sepal = LogisticRegression(max_iter=200)
model_sepal.fit(X_train_sepal, y_train_sepal)
y_pred_sepal = model_sepal.predict(X_test_sepal)

print("Sepal-only accuracy:", accuracy_score(y_test_sepal, y_pred_sepal))
print("Sepal-only confusion matrix:")
print(confusion_matrix(y_test_sepal, y_pred_sepal))

Let's push this further and try every single measurement completely on its own:

In [ ]:
for feature in iris.feature_names:
    X_one = df[[feature]]
    X_train_one, X_test_one, y_train_one, y_test_one = train_test_split(
        X_one, y, test_size=0.20, random_state=42, stratify=y
    )
    model_one = LogisticRegression(max_iter=200)
    model_one.fit(X_train_one, y_train_one)
    acc = model_one.score(X_test_one, y_test_one)
    print(f"{feature}: {acc:.4f}")

Petal measurements alone do almost as well as all four features combined; sepal measurements alone do
noticeably worse. That's not a coincidence — it's an early, informal taste of **feature importance**, which
we'll treat properly with real models in Week 10.

---

## Part 4 — Where Does It Go Wrong, and How Confident Is It?

Accuracy is a single number. It's worth actually looking at *which* flowers got misclassified, and how
confident the model was when it made each prediction.

In [ ]:
results = X_test_petal.copy()
results["actual"] = iris.target_names[y_test_petal]
results["predicted"] = iris.target_names[y_pred_petal]

results[results["actual"] != results["predicted"]]

In [ ]:
sns.scatterplot(
    data=results, x="petal length (cm)", y="petal width (cm)",
    hue="actual", style="predicted", s=120
)
plt.title("Actual vs. Predicted Species")
plt.show()

Logistic regression doesn't just output a guess — it outputs a probability for each class, and the
prediction is whichever probability is highest. Looking at those probabilities directly tells you something
accuracy alone never will: how *confident* the model was, even when it got the answer right.

In [ ]:
probabilities = model_petal.predict_proba(X_test_petal)

probability_df = pd.DataFrame(probabilities, columns=iris.target_names)
probability_df["actual"] = iris.target_names[y_test_petal]
probability_df["predicted"] = iris.target_names[y_pred_petal]
probability_df["confidence"] = probabilities.max(axis=1)

probability_df.sort_values("confidence").head(10)

The lowest-confidence rows are usually the borderline cases — flowers that look genuinely ambiguous
between two species. A model that's right but only 52% confident is telling you something very different
than a model that's right and 99.9% confident, even though both count identically toward "accuracy."

---

## Part 5 — Wait, Could We Use *Linear* Regression Here?

Good instinct to ask. Short answer: **not for predicting species** — but yes, if we change the question.

### Why not species?

`species` is **categorical**: setosa, versicolor, virginica. There's no numeric order between them — species
2 isn't "twice as much species" as species 1. Linear regression predicts a continuous *number*; asking it to
predict "1.4" for a flower and then rounding to the nearest species would be nonsense, and metrics like
accuracy or a confusion matrix don't even apply to a continuous output. This is the exact same
numerical-vs-categorical distinction from Lecture 1's Types of Data section — it's not just a vocabulary
exercise, it's the thing that determines which entire family of algorithm you reach for.

**The rule of thumb:** if the target is a *category*, you're doing classification. If the target is a
*continuous number*, you're doing regression. The algorithm's name doesn't always help — "logistic
regression" is a classification algorithm, confusingly enough (Part 2). What matters is the target.

### So when *can* we use linear regression on this dataset?

If we pick a different question — one with a continuous target instead of a categorical one. For example:

> Can petal length, petal width, and sepal width predict **sepal length**?

Now the target is a measurement in centimeters — continuous — so this is a genuine regression problem.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_reg = df[["sepal width (cm)", "petal length (cm)", "petal width (cm)"]]
y_reg = df["sepal length (cm)"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train_reg, y_train_reg)

y_pred_reg = linear_model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, y_pred_reg))
print("MSE:", mean_squared_error(y_test_reg, y_pred_reg))
print("R²:", r2_score(y_test_reg, y_pred_reg))

Notice the evaluation metrics changed completely — and that's not a random choice, it follows directly
from the target changing:

- **MAE** (mean absolute error): on average, how many centimeters off was each prediction? A very literal,
  interpretable number.
- **MSE** (mean squared error): like MAE, but squares each error first — which punishes big misses much
  more than small ones.
- **R² (R-squared)**: roughly, "what fraction of the variation in sepal length does the model explain?" 1.0
  would be a perfect fit; 0.0 would mean the model does no better than just guessing the average every time.

None of these would make any sense for the species-prediction problem in Part 2 — there's no "off by 0.3
species." And accuracy or a confusion matrix would make no sense here — there's no fixed set of categories
to be right or wrong about. **The target's data type doesn't just pick the algorithm family — it picks the
entire evaluation vocabulary that goes with it.** We'll go much deeper on regression metrics in Week 7 proper
and classification metrics in Week 9.

---

## Part 6 — Is One Test Split Enough to Trust?

We picked `random_state=42` for our train/test split earlier — but what if we'd happened to get an unlucky
split? **Cross-validation** answers this by splitting the data multiple different ways and averaging the
results. This is a preview of Week 11.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    LogisticRegression(max_iter=200), X_petal, y, cv=5, scoring="accuracy"
)

print("Fold accuracies:", scores)
print(f"Mean accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

`cv=5` means the data got split 5 different ways, and the model was trained and tested fresh each time.
The spread between fold accuracies (the standard deviation) tells you how much your single-split result
might have just been luck. A small spread here is, again, the toy-dataset effect — real data usually swings
more between folds.

---

## Part 7 — Logistic Regression Isn't the Only Option

Scikit-learn's whole design is that every model type shares the same `.fit()` / `.predict()` interface —
which makes comparing model families almost embarrassingly easy. This is a preview of Weeks 8 and 10.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "Logistic regression": LogisticRegression(max_iter=200),
    "K-nearest neighbors": KNeighborsClassifier(),
    "Decision tree": DecisionTreeClassifier(random_state=42),
    "Random forest": RandomForestClassifier(random_state=42),
    "Support vector machine": SVC(),
}

for name, candidate_model in models.items():
    scores = cross_val_score(candidate_model, X_petal, y, cv=5, scoring="accuracy")
    print(f"{name}: {scores.mean():.4f}")

Decision trees have a nice property the others don't: you can actually look at the decision rules it
learned.

In [ ]:
from sklearn.tree import plot_tree

tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train_petal, y_train_petal)

plt.figure(figsize=(12, 7))
plot_tree(
    tree_model,
    feature_names=X_petal.columns,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
)
plt.show()

---

## Recap

In one sitting, we touched: EDA, classification, train/test splits, accuracy and confusion matrices,
feature selection, prediction confidence, the regression/classification distinction, regression metrics,
cross-validation, and comparing five different model types. That's most of Weeks 7 through 11 in miniature.

It was fast and it was clean **because the data was fake-easy**: balanced classes, no missing values, no
ambiguous labels, obvious visual separation between groups. Starting next class, we go back to real NYC
data, and every one of these steps gets harder and more interesting for exactly that reason.